# Demo 02: YOLOE Deep Dive — Open Vocabulary Detection

**Block 4** | Day 1 | 20 minutes

This notebook demonstrates YOLOE's three prompting modes (text, visual, prompt-free)
and important failure cases that every practitioner needs to know.

**Key takeaways:**
- Open vocabulary = change what you detect by changing text, not retraining
- `set_classes()` is mandatory — skipping it silently returns zero detections
- Negation does not work in text prompts ("not wearing" = "wearing")
- Prompt wording has 2x+ impact on detection count

**Model:** YOLOE-26L-seg (44.9M params, 6.2ms on T4, 161 FPS)

In [ ]:
# Cell 1: Environment check and device detection
import torch
import sys
from pathlib import Path

# Device selection: CUDA > MPS > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_properties(0).name
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: {DEVICE} ({gpu_name}, {gpu_mem:.1f} GB)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print(f"Device: {DEVICE} (Apple Silicon)")
else:
    DEVICE = "cpu"
    print(f"Device: {DEVICE} (no GPU detected — demo will be slower)")

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

import supervision as sv
print(f"Supervision: {sv.__version__}")

# Data directory — try HPC layout first, then fall back to local workshop path
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data" / "synthetic_ppe"
if not DATA_DIR.exists():
    DATA_DIR = Path("/Users/dejandukic/dejan_dev/ai-factory/cv-workshop/workshop/data/synthetic_ppe")
    if not DATA_DIR.exists():
        raise FileNotFoundError(
            f"Cannot find synthetic_ppe data directory.\n"
            f"Tried: {WORKSHOP_DIR / 'data' / 'synthetic_ppe'}\n"
            f"Tried: {DATA_DIR}"
        )

# Verify we have images
test_images = sorted(DATA_DIR.rglob("*.jpg"))
print(f"\nData directory: {DATA_DIR}")
print(f"Images found: {len(test_images)}")
print(f"\nReady.")

---
## Part 1: Text Prompt Mode

The core workflow: load model, set text classes, predict.

**`set_classes()`** converts your text labels into embeddings via MobileCLIP.
Once set, every subsequent `predict()` call uses those classes.

In [ ]:
# Cell 3: Load YOLOE, set classes, run on easy PPE image
from ultralytics import YOLOE
from PIL import Image
import supervision as sv

# Load model
model = YOLOE("yoloe-26l-seg.pt")

# Set text classes — this is the key step
NAMES = ["helmet", "person"]
model.set_classes(NAMES, model.get_text_pe(NAMES))

# Run on easy image
image_path = str(DATA_DIR / "easy" / "easy_01.jpg")
image = Image.open(image_path)
results = model.predict(image, conf=0.3, verbose=False)

# Visualize with supervision
detections = sv.Detections.from_ultralytics(results[0])

annotated = image.copy()
annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.5, text_padding=3).annotate(
    scene=annotated, detections=detections
)

print(f"Found {len(detections)} objects:")
for i in range(len(detections)):
    cls_id = detections.class_id[i]
    conf = detections.confidence[i]
    cls_name = NAMES[cls_id] if cls_id < len(NAMES) else f"class_{cls_id}"
    print(f"  {cls_name}: {conf:.2f}")

annotated

In [ ]:
# Cell 4: Change classes — detect completely different things, same model
NEW_NAMES = ["safety vest", "person", "crane", "excavator"]
model.set_classes(NEW_NAMES, model.get_text_pe(NEW_NAMES))

image_path = str(DATA_DIR / "baseline" / "baseline_01.jpg")
image = Image.open(image_path)
results = model.predict(image, conf=0.25, verbose=False)

detections = sv.Detections.from_ultralytics(results[0])

annotated = image.copy()
annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.5, text_padding=3).annotate(
    scene=annotated, detections=detections
)

print(f"Classes: {NEW_NAMES}")
print(f"Found {len(detections)} objects:")
for i in range(len(detections)):
    cls_id = detections.class_id[i]
    conf = detections.confidence[i]
    cls_name = NEW_NAMES[cls_id] if cls_id < len(NEW_NAMES) else f"class_{cls_id}"
    print(f"  {cls_name}: {conf:.2f}")

print("\nSame model. Same weights. Different words. That's open vocabulary.")

annotated

In [ ]:
# Cell 5: Audience suggestion — type classes live during demo
#
# [LIVE INTERACTION] Ask the audience: "What do you want to detect?"
# Type their suggestions below and run this cell.
#

AUDIENCE_NAMES = ["person"]  # <-- Replace with audience suggestions
model.set_classes(AUDIENCE_NAMES, model.get_text_pe(AUDIENCE_NAMES))

image_path = str(DATA_DIR / "baseline" / "baseline_01.jpg")
image = Image.open(image_path)
results = model.predict(image, conf=0.25, verbose=False)

detections = sv.Detections.from_ultralytics(results[0])

annotated = image.copy()
annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.5, text_padding=3).annotate(
    scene=annotated, detections=detections
)

print(f"Audience classes: {AUDIENCE_NAMES}")
print(f"Found {len(detections)} objects")

annotated

In [ ]:
# Cell 6: Run on a harder image — edge cases
NAMES = ["helmet", "person"]
model.set_classes(NAMES, model.get_text_pe(NAMES))

image_path = str(DATA_DIR / "edge_cases" / "edge_cases_01.jpg")
image = Image.open(image_path)
results = model.predict(image, conf=0.25, verbose=False)

detections = sv.Detections.from_ultralytics(results[0])

annotated = image.copy()
annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.5, text_padding=3).annotate(
    scene=annotated, detections=detections
)

print(f"Found {len(detections)} objects on a harder image:")
for i in range(len(detections)):
    cls_id = detections.class_id[i]
    conf = detections.confidence[i]
    cls_name = NAMES[cls_id] if cls_id < len(NAMES) else f"class_{cls_id}"
    print(f"  {cls_name}: {conf:.2f}")

print("\nNotice the lower confidence scores and potential missed detections.")
print("Easy images are easy. Edge cases reveal the model's limits.")

annotated

---
## Part 2: Visual Prompt Mode (OPTIONAL — skip if behind)

Instead of text, give the model a reference image crop.
It finds everything that looks similar.

Visual prompting is rougher than text prompting — one reference image gives less
information than a text description backed by 400M image-caption pairs. But for
proprietary objects that don't exist in any training data, this may be your only option.

In [ ]:
# Cell 8: Visual prompt concept
# This cell demonstrates the visual prompt API.
# Skip this cell if running behind schedule.

import numpy as np
import cv2
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe.predict_vp import YOLOEVPSegPredictor

# Load fresh model for visual prompt mode
model_vp = YOLOE("yoloe-26l-seg.pt")

# Source image: draw a bounding box around a helmet to use as reference
source_path = str(DATA_DIR / "easy" / "easy_01.jpg")
source_image = Image.open(source_path)

# Define a reference bounding box for a helmet region
# Format: [x_min, y_min, x_max, y_max]
# Adjust these coordinates based on the actual image content
VP_NAMES = ["helmet"]
bboxes = np.array([[200, 50, 300, 150]], dtype=np.float64)  # Approximate helmet region
cls = np.array([0], dtype=np.int32)  # Class index for "helmet"

prompts = dict(bboxes=bboxes, cls=cls)

# First pass: extract visual prompt embeddings from the source image
model_vp.predict(source_image, prompts=prompts, predictor=YOLOEVPSegPredictor, return_vpe=True)
model_vp.set_classes(VP_NAMES, model_vp.predictor.vpe)
model_vp.predictor = None

# Second pass: use the visual prompt to detect on a different image
target_path = str(DATA_DIR / "baseline" / "baseline_01.jpg")
target_image = Image.open(target_path)

results = model_vp.predict(target_image, conf=0.25, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])

annotated = target_image.copy()
annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.5, text_padding=3).annotate(
    scene=annotated, detections=detections
)

print(f"Visual prompt: found {len(detections)} helmet-like objects")
print("\nVisual prompting: 'find things that look like this crop.'")
print("One reference image instead of text. Rougher, but works for novel objects.")

del model_vp
annotated

---
## Part 3: Prompt-Free Mode

No prompt at all. YOLOE has **4,585 built-in categories** from LVIS.
It detects everything it recognizes.

Lower accuracy (27.2 mAP vs 36.8 for text prompting), but useful for
exploratory analysis: *what's actually in this image?*

In [ ]:
# Cell 10: Prompt-free detection
from ultralytics import YOLOE

# Load the prompt-free variant (different weights)
model_pf = YOLOE("yoloe-26l-seg-pf.pt")

image_path = str(DATA_DIR / "baseline" / "baseline_01.jpg")
image = Image.open(image_path)

# No set_classes() needed — prompt-free mode uses built-in LVIS vocabulary
results = model_pf.predict(image, conf=0.15, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])

# List all detected categories
found_classes = {}
for i in range(len(detections)):
    cls_id = int(detections.class_id[i])
    cls_name = model_pf.names.get(cls_id, f"class_{cls_id}")
    conf = float(detections.confidence[i])
    if cls_name not in found_classes:
        found_classes[cls_name] = []
    found_classes[cls_name].append(conf)

print(f"Prompt-free mode: found {len(detections)} objects across {len(found_classes)} categories:\n")
for cls_name in sorted(found_classes.keys()):
    confs = found_classes[cls_name]
    avg_conf = sum(confs) / len(confs)
    print(f"  {cls_name}: {len(confs)} detection(s), avg conf {avg_conf:.2f}")

print(f"\nNo prompt given. The model tried all 4,585 LVIS categories.")
print(f"Some make sense. Some are... creative.")

# Visualize
annotated = image.copy()
annotated = sv.BoxAnnotator(thickness=2).annotate(scene=annotated, detections=detections)
annotated = sv.LabelAnnotator(text_scale=0.4, text_padding=2).annotate(
    scene=annotated, detections=detections
)

del model_pf
annotated

---
## Part 4: Failure Cases and Gotchas

These are the things that will cost you hours of debugging if you don't know about them.
Every one of these was discovered the hard way.

In [ ]:
# Cell 12: THE GOTCHA — no set_classes() = zero detections, NO error
#
# This is the most common mistake with YOLOE.
# If you forget set_classes(), the model silently returns nothing.

from ultralytics import YOLOE

# Load model fresh — deliberately do NOT call set_classes()
model_fresh = YOLOE("yoloe-26l-seg.pt")

# Run inference without setting classes first
image_path = str(DATA_DIR / "easy" / "easy_01.jpg")
image = Image.open(image_path)
results = model_fresh.predict(image, conf=0.3, verbose=False)

detections = sv.Detections.from_ultralytics(results[0])

print("=" * 60)
print("GOTCHA: Fresh model, no set_classes()")
print("=" * 60)
print(f"Detections: {len(detections)}")
print(f"Errors: None")
print(f"Warnings: None")
print()
print("Zero detections. No error. No warning. No exception.")
print("The model silently returns nothing.")
print()
print("ALWAYS call set_classes() before inference.")
print("Or use the prompt-free variant (yoloe-*-pf.pt) explicitly.")

del model_fresh

In [ ]:
# Cell 13: The negation paradox
#
# "person wearing a hard hat" vs "person NOT wearing a hard hat"
# Intuition says these should find opposite things. They don't.

from ultralytics import YOLOE

model_neg = YOLOE("yoloe-26l-seg.pt")

image_path = str(DATA_DIR / "baseline" / "baseline_01.jpg")
image = Image.open(image_path)

# Prompt A: person wearing a hard hat
names_a = ["person wearing a hard hat"]
model_neg.set_classes(names_a, model_neg.get_text_pe(names_a))
results_a = model_neg.predict(image, conf=0.25, verbose=False)
dets_a = sv.Detections.from_ultralytics(results_a[0])

# Prompt B: person NOT wearing a hard hat
names_b = ["person not wearing a hard hat"]
model_neg.set_classes(names_b, model_neg.get_text_pe(names_b))
results_b = model_neg.predict(image, conf=0.25, verbose=False)
dets_b = sv.Detections.from_ultralytics(results_b[0])

print("=" * 60)
print("NEGATION PARADOX")
print("=" * 60)
print(f"'person wearing a hard hat':     {len(dets_a)} detections")
print(f"'person NOT wearing a hard hat':  {len(dets_b)} detections")
print()
print("Same number. Same detections. 'NOT' is invisible to the model.")
print()
print("Why? Only 0.08% of internet captions contain 'not'.")
print("The model never learned what negation means.")
print("Both prompts land in the same region of embedding space.")
print()
print("Practical consequence: you CANNOT encode compliance in a prompt.")
print("Detect the THINGS (person, helmet), check the RELATIONSHIP with code.")

# Show both results side by side
ann_a = image.copy()
ann_a = sv.BoxAnnotator(thickness=2).annotate(scene=ann_a, detections=dets_a)
ann_a = sv.LabelAnnotator(text_scale=0.4, text_padding=2).annotate(
    scene=ann_a, detections=dets_a
)

ann_b = image.copy()
ann_b = sv.BoxAnnotator(thickness=2).annotate(scene=ann_b, detections=dets_b)
ann_b = sv.LabelAnnotator(text_scale=0.4, text_padding=2).annotate(
    scene=ann_b, detections=dets_b
)

sv.plot_images_grid(
    images=[ann_a, ann_b],
    grid_size=(1, 2),
    titles=['"person wearing a hard hat"', '"person NOT wearing a hard hat"'],
    size=(14, 6),
)

del model_neg

In [ ]:
# Cell 14: Prompt sensitivity — word choice matters more than you think
#
# In our experiments across 75 images:
#   "helmet" found 2.2x more helmets than "hard hat, safety helmet"
# Three words. 2.2x difference.

from ultralytics import YOLOE

model_ps = YOLOE("yoloe-26l-seg.pt")

image_path = str(DATA_DIR / "baseline" / "baseline_01.jpg")
image = Image.open(image_path)

# Prompt A: specific, domain-accurate terminology
names_a = ["hard hat", "safety helmet"]
model_ps.set_classes(names_a, model_ps.get_text_pe(names_a))
results_a = model_ps.predict(image, conf=0.25, verbose=False)
dets_a = sv.Detections.from_ultralytics(results_a[0])

# Prompt B: simple, generic word
names_b = ["helmet"]
model_ps.set_classes(names_b, model_ps.get_text_pe(names_b))
results_b = model_ps.predict(image, conf=0.25, verbose=False)
dets_b = sv.Detections.from_ultralytics(results_b[0])

print("=" * 60)
print("PROMPT SENSITIVITY")
print("=" * 60)
print(f"Prompt ['hard hat', 'safety helmet']: {len(dets_a)} detections")
print(f"Prompt ['helmet']:                    {len(dets_b)} detections")
print()
if len(dets_a) > 0:
    ratio = len(dets_b) / len(dets_a)
    print(f"Ratio: {ratio:.1f}x difference from changing the prompt word")
print()
print("'helmet' sits at the center of a broader concept cluster in")
print("embedding space — internet captions use 'helmet' more often")
print("than 'hard hat'. Broader cluster = more matches.")
print()
print("Across 75 PPE images: 'helmet' found 2.2x more detections")
print("than 'hard hat, safety helmet'. At higher confidence.")
print()
print("Prompt engineering is not a nice-to-have. It's a first-order lever.")

# Show both results side by side
ann_a = image.copy()
ann_a = sv.BoxAnnotator(thickness=2).annotate(scene=ann_a, detections=dets_a)
ann_a = sv.LabelAnnotator(text_scale=0.4, text_padding=2).annotate(
    scene=ann_a, detections=dets_a
)

ann_b = image.copy()
ann_b = sv.BoxAnnotator(thickness=2).annotate(scene=ann_b, detections=dets_b)
ann_b = sv.LabelAnnotator(text_scale=0.4, text_padding=2).annotate(
    scene=ann_b, detections=dets_b
)

sv.plot_images_grid(
    images=[ann_a, ann_b],
    grid_size=(1, 2),
    titles=['"hard hat" + "safety helmet"', '"helmet"'],
    size=(14, 6),
)

del model_ps

---
## Performance Numbers

### YOLOE Model Variants

| Variant | Params | mAP (LVIS text) | Speed (T4) | FPS |
|---------|--------|-----------------|------------|-----|
| YOLOE-S | 21.8M | 27.6 | 3.3ms | ~303 |
| YOLOE-M | 38.1M | 33.7 | 5.1ms | ~196 |
| YOLOE-L | 44.9M | 36.8 | 6.2ms | ~161 |

### Speed Comparison

| Model | Speed (T4) | Open-Vocab? |
|-------|------------|-------------|
| Grounding DINO | 30-50ms | Yes |
| **YOLOE-L** | **6.2ms** | **Yes** |
| YOLO26-L | 6.2ms | No |
| YOLO26-N | 1.7ms | No |

YOLOE is **5-10x faster** than Grounding DINO with equivalent or better accuracy.

But YOLO26-N is **3.6x faster** than YOLOE — and it runs on edge devices.
That speed gap is why **distillation** exists.

---

**Next:** Block 5 — Distillation. Take everything YOLOE knows about helmets and people,
and pour it into a tiny YOLO26-N model that runs at 1.7ms on edge hardware.